# Visual test: event-time plot functions

Tests `time_to_X_log`, `time_to_X_linear`, `X_timecourse_log`, `X_timecourse_linear`, and `save_event_time_plot_set` from `analysis_style.py` using synthetic data with known properties.

**Expected behaviour by section:**
- **nth-event plots** — Fast condition (λ=5s) should lie _below_ Slow (λ=15s): it reaches the nth event sooner.
- **Timecourse plots** — Fast condition should rise _faster_: more cumulative events by any time t.
- **Log variants** — should show per-episode traces (thin semi-transparent lines) in addition to the mean±SEM band.
- **Linear variants** — mean±SEM band only, no traces.
- **`min_episodes` filter** — the `Too Few` condition (2 episodes) should be silently suppressed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'fish') if os.getcwd().endswith('notebooks') else os.getcwd())

import numpy as np
from analysis_style import (
    inline_display,
    time_to_X_log,
    time_to_X_linear,
    X_timecourse_log,
    X_timecourse_linear,
    save_event_time_plot_set,
)

print('Imports OK')

In [ ]:
# ── synthetic data ─────────────────────────────────────────────────────────
rng = np.random.default_rng(42)

def make_episodes(n_episodes, mean_iet, n_events, rng):
    """Poisson process: return list of cumulative-time arrays, one per episode."""
    return [np.cumsum(rng.exponential(scale=mean_iet, size=n_events))
            for _ in range(n_episodes)]

N_EP   = 20
fast_eps = make_episodes(N_EP, mean_iet=5,  n_events=30, rng=rng)
slow_eps = make_episodes(N_EP, mean_iet=15, n_events=12, rng=rng)
tiny_eps = make_episodes(2,    mean_iet=8,  n_events=20, rng=rng)  # < min_episodes=3

palette = {'Fast (λ=5s)': '#4477AA', 'Slow (λ=15s)': '#EE6677', 'Too Few': '#228833'}

condition_times_2 = [('Fast (λ=5s)', fast_eps), ('Slow (λ=15s)', slow_eps)]
condition_times_3 = condition_times_2 + [('Too Few', tiny_eps)]

print(f'Fast: mean last event ~{np.mean([e[-1] for e in fast_eps]):.1f}s')
print(f'Slow: mean last event ~{np.mean([e[-1] for e in slow_eps]):.1f}s')
print(f'Too Few: 2 episodes (min_episodes=3 → suppressed)')

## 1. `time_to_X_log` — nth event vs time, log Y, with per-episode traces

Check: Fast (blue) should be **below** Slow (red) at every nth event. Thin traces should be visible.

In [ ]:
with inline_display():
    time_to_X_log(condition_times_2, palette, 'time_to_x_log.pdf',
                  event_label='Food consumed (n)', time_label='Time to nth food (s)')

## 2. `time_to_X_linear` — nth event vs time, linear Y, mean±SEM only

Check: same ordering as above; no per-episode traces; SEM ribbons visible.

In [ ]:
with inline_display():
    time_to_X_linear(condition_times_2, palette, 'time_to_x_linear.pdf',
                     event_label='Food consumed (n)', time_label='Time to nth food (s)')

## 3. `X_timecourse_log` — cumulative count over time, log X, with traces

Check: Fast (blue) rises **above** Slow (red) at every timepoint. Log time axis. Thin traces visible.

In [ ]:
with inline_display():
    X_timecourse_log(condition_times_2, palette, 'x_timecourse_log.pdf',
                     event_label='Food consumed (n)', time_label='Time (s)')

## 4. `X_timecourse_linear` — cumulative count over time, linear X, mean±SEM only

Check: same ordering; linear time axis; no traces; SEM ribbons visible.

In [ ]:
with inline_display():
    X_timecourse_linear(condition_times_2, palette, 'x_timecourse_linear.pdf',
                        event_label='Food consumed (n)', time_label='Time (s)')

## 5. `min_episodes` filtering

Adding `Too Few` (2 episodes, default `min_episodes=3`). The green line should **not appear**.

In [ ]:
with inline_display():
    time_to_X_log(condition_times_3, palette, 'filter_test_log.pdf',
                  event_label='Food consumed (n)', time_label='Time to nth food (s)')

## 6. `save_event_time_plot_set` — all four variants in one call

Should produce four figures (log, linear, timecourse_log, timecourse_linear). Requires ≥2 conditions.

In [ ]:
with inline_display():
    save_event_time_plot_set(condition_times_2, palette, out_dir='.',
                             stem='consumption', event_label='Food consumed (n)',
                             time_label='Time (s)')

## 7. Edge case: single condition (legend suppressed)

`save_event_time_plot_set` skips when len < 2. Direct calls with one condition should plot without a legend.

In [ ]:
with inline_display():
    time_to_X_linear([('Fast only', fast_eps)], {'Fast only': '#4477AA'},
                     'single_cond_linear.pdf',
                     event_label='Food consumed (n)', time_label='Time to nth food (s)')

# save_event_time_plot_set with 1 condition should silently skip (no output)
with inline_display():
    save_event_time_plot_set([('Fast only', fast_eps)], {'Fast only': '#4477AA'},
                             out_dir='.', stem='single_skipped')
print('save_event_time_plot_set with 1 condition: silently skipped (no figure above)')